In [ ]:
api="L5LGF7NHJ4SZ0Q9N"

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import requests
from pathlib import Path

CACHE_DIR = Path("data/cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)


def get_alpha_vantage_candles(
    symbol: str,
    api_key: str,
    days: int = 365,
    timeout: int = 20,
    trust_env: bool = False,
) -> pd.DataFrame:
    """
    Télécharge l'historique OHLCV via Alpha Vantage.
    
    Plan gratuit : 25 requêtes/jour, 5 requêtes/minute.
    outputsize='full' pour l'historique complet (20+ ans).
    
    Args:
        symbol: Ticker (ex: 'AAPL', 'MSFT').
        api_key: Clé API Alpha Vantage (gratuite sur alphavantage.co).
        days: Nombre de jours d'historique (max ~7300 pour 'full').
        timeout: Timeout en secondes.
        trust_env: Utiliser le proxy système (False par défaut).
    
    Returns:
        DataFrame avec colonnes open, high, low, close, volume et index date.
    """
    symbol = symbol.upper()
    cache_file = CACHE_DIR / f"{symbol}_AV_{days}d.csv"

    # Alpha Vantage ne supporte pas 'days' directement → on prend 'full' et on filtre
    params = {
        "function": "TIME_SERIES_DAILY",
        "symbol": symbol,
        "outputsize": "full",
        "apikey": api_key,
    }

    try:
        with requests.Session() as session:
            session.trust_env = trust_env
            response = session.get(
                "https://www.alphavantage.co/query",
                params=params,
                timeout=timeout,
            )

        if response.status_code == 403:
            raise PermissionError(
                f"Accès refusé (HTTP 403) pour {symbol}. Vérifie ta clé API."
            )
        if response.status_code == 429:
            raise RuntimeError(
                "Limite de requêtes atteinte (HTTP 429). Attends 1 minute."
            )

        response.raise_for_status()
        payload = response.json()

        # Vérifier les erreurs Alpha Vantage
        if "Error Message" in payload:
            raise ValueError(f"Erreur API : {payload['Error Message']}")
        if "Note" in payload:
            raise RuntimeError(f"Rate limit : {payload['Note']}")
        if "Time Series (Daily)" not in payload:
            raise ValueError(f"Données non disponibles pour {symbol}")

        # Parser la réponse
        ts = payload["Time Series (Daily)"]
        df = pd.DataFrame.from_dict(ts, orient='index')
        df = df.rename(columns={
            '1. open': 'open', '2. high': 'high', '3. low': 'low',
            '4. close': 'close', '5. volume': 'volume'
        })
        df.index = pd.to_datetime(df.index)
        df = df.sort_index()
        df = df.astype(float)

        # Filtrer sur les 'days' derniers jours
        end_dt = pd.Timestamp.now(tz='UTC').tz_localize(None)
        start_dt = end_dt - pd.Timedelta(days=days)
        df = df[df.index >= start_dt]

        # Sauvegarder en cache
        df.to_csv(cache_file)
        return df[['open', 'high', 'low', 'close', 'volume']]

    except Exception as exc:
        print(f"[Avertissement] Alpha Vantage indisponible ({exc}). Repli cache/fallback...")
        if cache_file.exists():
            cached = pd.read_csv(cache_file, index_col="date", parse_dates=True)
            return cached

        # Fallback déterministe hors ligne (GBM)
        dates = pd.date_range(end=pd.Timestamp.now(), periods=252, freq="B")
        rng = np.random.default_rng(42)
        rets = rng.normal(loc=0.0003, scale=0.012, size=len(dates))
        close = 100.0 * np.exp(np.cumsum(rets))
        return pd.DataFrame({"close": close}, index=dates)


def compute_historical_volatility(df: pd.DataFrame, trading_days: int = 3) -> float:
    """Calcule la volatilité annualisée à partir de la série 'close'."""
    returns = df["close"].pct_change().dropna()
    if returns.empty:
        return 0.20
    return float(returns.std() * np.sqrt(trading_days))


if __name__ == "__main__":
    # Clé API : variable d'environnement ou valeur par défaut
    print('debut')
    api_key = os.getenv("ALPHAVANTAGE_API_KEY", "L5LGF7NHJ4SZ0Q9N").strip()
    print("post api")
    if not api_key:
        print("[Info] Variable ALPHAVANTAGE_API_KEY non définie.")
        print("Obtiens une clé gratuite sur : https://www.alphavantage.co/support/#api-key")
        api_key = input("Entre ta clé API : ").strip()

    print("pre symbole")
# input("Ticker (ex: AAPL) : ").strip()
    symbol = "AAPL"

    df = get_alpha_vantage_candles(symbol, api_key=api_key, days=3, trust_env=False)
    print("post candle")
    sigma = compute_historical_volatility(df)
    print("post volatility")
    spot = float(df["close"].iloc[-1])

    print(f"\nLignes chargées : {len(df)}")
    print(f"Spot (S) : {spot:.2f} | Volatilité historique (σ) : {sigma:.2%}")
    print(f"Période : {df.index[0].date()} → {df.index[-1].date()}")
    print(df.tail(5))

debut
post api
pre symbole


In [20]:
import yfinance as yf
import pandas as pd
import numpy as np

def get_yahoo_candles(symbol: str, days: int = 365) -> pd.DataFrame:
    """Télécharge l'historique OHLCV via Yahoo Finance (gratuit)."""
    ticker = yf.Ticker(symbol)
    df = ticker.history(period=f"{days}d")
    df = df.rename(columns={
        'Open': 'open', 'High': 'high', 'Low': 'low',
        'Close': 'close', 'Volume': 'volume'
    })
    return df[['open', 'high', 'low', 'close', 'volume']]

# Test
df = get_yahoo_candles('AAPL', days=365)
print(f"Lignes : {len(df)}")
print(f"Spot : {df['close'].iloc[-1]:.2f}")
sigma = df['close'].pct_change().dropna().std() * np.sqrt(252)
print(f"Volatilité : {sigma:.2%}")

Cookie/crumb fetch failed (Timeout), continuing without crumb
Failed to get ticker 'AAPL' reason: Failed to perform, curl: (28) Connection timed out after 10016 milliseconds. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.
Cookie/crumb fetch failed (ConnectionError), continuing without crumb


Timeout: Failed to perform, curl: (28) Connection timed out after 30047 milliseconds. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.

In [18]:
import requests
import pandas as pd

def get_alpha_vantage_candles(symbol: str, api_key: str, days: int = 365) -> pd.DataFrame:
    """Télécharge l'historique OHLCV via Alpha Vantage (gratuit avec clé)."""
    url = "https://www.alphavantage.co/query"
    params = {
        "function": "TIME_SERIES_DAILY",
        "symbol": symbol,
        "outputsize": "full",
        "apikey": api_key
    }
    response = requests.get(url, params=params, timeout=20)
    data = response.json()
    
    if "Time Series (Daily)" not in data:
        raise ValueError(f"Données non disponibles pour {symbol}")
    
    df = pd.DataFrame.from_dict(data["Time Series (Daily)"], orient='index')
    df = df.rename(columns={
        '1. open': 'open', '2. high': 'high', '3. low': 'low',
        '4. close': 'close', '5. volume': 'volume'
    })
    df.index = pd.to_datetime(df.index)
    df = df.sort_index()
    df = df.astype(float)
    return df[['open', 'high', 'low', 'close', 'volume']]